## Scraping Health-Related Forums and Articles

This project focuses on extracting posts and articles from health-related forums using web scraping techniques. These forums host a wide range of user-generated content on diverse health topics, making them valuable for applications like sentiment analysis, trend monitoring, and topic modeling.

### Objectives
- **Data Collection**: Retrieve posts and articles from specific health-related categories based on topic criteria (e.g., mental health, fitness).
- **Data Processing**: Clean and preprocess the extracted data to prepare it for analysis.
- **Data Storage**: Store the scraped data in a structured format, such as CSV or a database, for further analysis.

### Tools and Technologies
- **Python**: The primary programming language for web scraping.
- **Beautiful Soup**: A library for parsing HTML and extracting data.
- **Requests**: A library for making HTTP requests to access web pages.
- **Pandas**: A data manipulation library to handle and analyze the scraped data.

### Getting Started
1. **Set Up the Environment**: Install the necessary libraries using `pip`.
2. **Define Scraping Logic**: Write functions to scrape data from specific health categories on the forum.
3. **Run the Scraper**: Execute the scraping script and monitor the data collection process.
4. **Analyze the Data**: Use Pandas to analyze the collected posts and articles for insights.

### Conclusion
This project provides hands-on experience in web scraping and data analysis using Python, leveraging real-world data from online health communities for deeper insights.


<p style="color:#FE4406;text-align:center;font-size:30px"> Scraping PLOS Articles</p>

In [2]:
#!pip install bs4
#!pip install selenium

### Scraping PLOS Articles 

In [3]:
## importing libraries
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time
from bs4 import BeautifulSoup
from urllib.request import urlopen, Request


<p style="color:#FFC107;text-align:left;font-size:20px"> Searching for PLOS's health related Articles   </p>

In [4]:
from bs4 import BeautifulSoup
from urllib.request import urlopen, Request
import csv
from concurrent.futures import ThreadPoolExecutor, as_completed


In [5]:
def save_articles_to_file(index, articles):

    with open(f"./data/pmcArticles{index}.csv", "w", encoding="utf-8", newline="") as f:

        writer = csv.writer(f)
        writer.writerow(["Authors", "Article Title", "Article Link"])

        for article in articles:
            writer.writerow([
                article["authors"],
                article["articleTitle"],
                article["articleLink"]
            ])
    print("a periodic save has been done ! ")


In [6]:
import requests
import time 
BASE_URL = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"


def scrape_page(page):

    params = {
        "query": "SRC:*",
        "pageSize": 150,
        "page": page,
        "format": "json"
    }

    r = requests.get(BASE_URL, params=params)
    data = r.json()

    page_articles = []

    for article in data["resultList"]["result"]:

        page_articles.append({
            "authors": article.get("authorString"),
            "articleTitle": article.get("title"),
            "articleLink": f"https://europepmc.org/article/{article.get('source')}/{article.get('id')}"
        })

    print(f"Scraped page {page} -> {len(page_articles)} articles")
    save_articles_to_file(str(time.time()).split(".")[0], page_articles)
    return page_articles

In [11]:
from concurrent.futures import ThreadPoolExecutor, as_completed


def main():

    pages_to_scrape = range(9996 , 150000 )

    with ThreadPoolExecutor(max_workers=15) as executor:

        futures = {
            executor.submit(scrape_page, page): page
            for page in pages_to_scrape
        }

        for future in as_completed(futures):

            page = futures[future]
            result = future.result()

            

    print("Scraping finished")

In [ ]:
main()

Scraped page 10001 -> 150 articles
Scraped page 9998 -> 150 articles
Scraped page 10005 -> 150 articles
a periodic save has been done ! 
a periodic save has been done ! 
a periodic save has been done ! 
Scraped page 10010 -> 150 articles
a periodic save has been done ! 
Scraped page 10009 -> 150 articles
a periodic save has been done ! 
Scraped page 9999 -> 150 articles
Scraped page 10002 -> 150 articles
a periodic save has been done ! 
a periodic save has been done ! 
Scraped page 9996 -> 150 articles
a periodic save has been done ! 
Scraped page 10008 -> 150 articles
a periodic save has been done ! 
Scraped page 10007 -> 150 articles
Scraped page 10006 -> 150 articles
a periodic save has been done ! 
a periodic save has been done ! 
Scraped page 10003 -> 150 articles
Scraped page 9997 -> 150 articles
a periodic save has been done ! 
a periodic save has been done ! 
Scraped page 10018 -> 150 articles
a periodic save has been done ! 
Scraped page 10012 -> 150 articles
a periodic save h

In [1]:
import os
import pandas as pd

def merge_csv_in_batches(folder_path, batch_size, prefix="merged"):
    files = sorted([f for f in os.listdir(folder_path) if f.endswith(".csv")])
    batch_number = 1

    for i in range(0, len(files), batch_size):
        batch_files = files[i:i + batch_size]
        merged_file_path = os.path.join(folder_path, f"{prefix}_{batch_number}.csv")
        
        # Read and concatenate all CSVs in the batch
        df_list = []
        for idx, file_name in enumerate(batch_files):
            file_path = os.path.join(folder_path, file_name)
            if idx == 0:
                # Keep header of the first file
                df = pd.read_csv(file_path)
            else:
                # Skip header for the next files
                df = pd.read_csv(file_path, header=0)
            df_list.append(df)
            os.remove(file_path)  # delete original file
        
        merged_df = pd.concat(df_list, ignore_index=True)
        merged_df.to_csv(merged_file_path, index=False)
        print(f"Created {merged_file_path} from {len(batch_files)} files")
        batch_number += 1

In [2]:
merge_csv_in_batches("./data", batch_size=941)

Created ./data\merged_1.csv from 110 files


In [4]:
import pandas as pd 
data = pd.read_csv("./data/data.csv")
print(len(data))

114000
